# Esperimento di Topic Labeling

In questo notebook confrontiamo diversi approcci per l'etichettatura automatica dei cluster individuati tramite HDBSCAN.

## Obiettivi:
1. Caricare i cluster e i documenti associati.
2. Applicare **YAKE**, **TextRank**, **c-TF-IDF** e **KeyBERT (Simulated)**.
3. Estrarre i documenti rappresentativi per ogni cluster.
4. Salvare i risultati comparativi nei metadata.

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import json
import re
from sentence_transformers import SentenceTransformer

# Aggiungiamo src al path per importare le utility
sys.path.append(os.path.abspath("../../"))

from src.utils.topic_labeling import (
    extract_keywords_yake, 
    extract_keywords_textrank, 
    calculate_ctfidf, 
    extract_keywords_keybert,
    get_cluster_representative_docs
)

## 1. Caricamento Dati

In [2]:
df_processed = pd.read_parquet("../../data/processed/jmail_emails_processed.parquet")
df_clusters = pd.read_parquet("../../data/processed/email_cluster_assignments.parquet")
embeddings = np.load("../../data/embeddings/email_embeddings_bge-small-en-v1-5.npy")

df = df_processed.merge(df_clusters, on="id", how="inner")
print(f"Totale righe: {len(df)}")

Totale righe: 42471


## 2. Applicazione Multi-Algoritmo
Eseguiamo tutti i metodi per ogni cluster.

In [3]:
from tqdm.notebook import tqdm

# Carichiamo il modello di embedding per KeyBERT
print("Caricamento del modello di embedding...")
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

docs_per_cluster = df.groupby("cluster_id")["combined_text"].apply(lambda x: " ".join(x)).to_dict()
cluster_list = sorted([c for c in docs_per_cluster.keys()])

# Pre-calcolo c-TF-IDF
print("Calcolo c-TF-IDF globale...")
ctfidf_labels_dict = calculate_ctfidf(docs_per_cluster, top_n=10)

final_labels = {}

print("Inizio estrazione labels per ogni cluster...")
for cid in tqdm(cluster_list):
    cluster_df = df[df["cluster_id"] == cid]
    
    # 1. Troviamo i documenti più rappresentativi per il sample text
    cluster_indices = cluster_df["embedding_row"].values
    cluster_embs = embeddings[cluster_indices]
    cluster_emb_centroid = cluster_embs.mean(axis=0)
    
    rep_docs = get_cluster_representative_docs(cluster_df["combined_text"].tolist(), cluster_embs, cluster_emb_centroid, n=50)
    sample_text = " ".join(rep_docs)
    
    # 2. Generiamo i veri Word Embeddings per KeyBERT
    words_in_sample = list(set(re.findall(r"\b[a-z]{3,}\b", sample_text.lower())))
    if len(words_in_sample) > 300:
        words_in_sample = words_in_sample[:300]
        
    # Se non ci sono parole valide (es. cluster di rumore senza testo utile)
    if not words_in_sample:
        final_labels[str(cid)] = {
            "ctfidf": ctfidf_labels_dict.get(cid, []),
            "yake": [],
            "textrank": [],
            "keybert_sim": []
        }
        continue
        
    word_embs_array = embedding_model.encode(words_in_sample, show_progress_bar=False)
    real_word_embs = {w: emb for w, emb in zip(words_in_sample, word_embs_array)}
    
    # 3. Applichiamo tutti gli algoritmi
    final_labels[str(cid)] = {
        "ctfidf": ctfidf_labels_dict.get(cid, []),
        "yake": extract_keywords_yake(sample_text, top_n=10),
        "textrank": extract_keywords_textrank(sample_text, top_n=10),
        "keybert_sim": extract_keywords_keybert(rep_docs[:10], cluster_emb_centroid, real_word_embs, top_n=10)
    }

print("Labeling completato.")

Caricamento del modello di embedding...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Calcolo c-TF-IDF globale...
Inizio estrazione labels per ogni cluster...


  0%|          | 0/13 [00:00<?, ?it/s]

/home/filippo/Scrivania/pizza-cluster/.venv/lib/python3.12/site-packages
Labeling completato.


## 3. Salvataggio Metadata

In [4]:
output_path = "../../data/metadata/cluster_labeling_metadata.json"
metadata = {
    "generated_at": pd.Timestamp.now().isoformat(),
    "algorithms": ["c-TF-IDF", "YAKE", "TextRank", "KeyBERT-Sim"],
    "n_clusters": len(final_labels),
    "cluster_labels": final_labels
}

with open(output_path, "w") as f:
    json.dump(metadata, f, indent=4)
print(f"Metadata aggiornati in {output_path}")

Metadata aggiornati in ../../data/metadata/cluster_labeling_metadata.json


## 4. Visualizzazione Comparativa

In [5]:
comparison_df = []
for cid in final_labels.keys():
    row = {"cluster": cid}
    row.update(final_labels[cid])
    comparison_df.append(row)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(comparison_df))


,cluster,ctfidf,yake,textrank,keybert_sim
0,-1,"[deutsche, td, bank, communication, com, confidential, new, epstein, york, db]","[Subject, Bank, Deutsche, communication, York, Kahn, Confidential, Richard, Jeffrey, Classification]","[| | | |Bergander | | | | | | Sent, | | | |, | | | | work, | | |, | | | | |Warner, | |, | Ticker | | Price, Client Contact | | | Richard Kahn, | | cell, | | |sent]","[jpmorgan, askvolume, mailing, hsbchalbis, updated, transfers, onboarded, amounts, accounts, ask]"
1,0,"[times, newyorktimesinfo, com, new, nytimes, york, http, digital, offer, subscription]","[Times, YORK, Digital, Offer, Subscription, time, Special, weeks, NYT, REDEEM]","[New York Times, New York Times digital subscriptions, New York Times Crossword, Times subscription, Times subscriptions, Times Digital Subscription, Times Subscription, Times, Times readers, Times Premier content]","[nytimes, nyt, subscribe, subscriber, expires, nytstore, columnists, emails, mails, readers]"
2,1,"[wikisource, senator, alberto, statement, gonzales, barack, library, nomination, attorney, general]","[Wikisource, Floor, General, Statement, Senator, Barack, Obama, Nomination, Alberto, Gonzales]","[Senator Barack Obama, Alberto Gonzales, Barack Obama, Floor Statement, the free online library, Mid-October, Attorney General - Wikisource, the free online library\n\nDate, the free online library\n\nNumber, the free online library\n\neric]","[obama, barack, gonzales, nomination, senator, statement, attorney, floor, whose, october]"
3,2,"[case, assigned, salesforce, assignment, notification, annual, link, description, click, customer]","[Case, Jeffrey, Epstein, CONFIDENTIAL, KYC, Description, ASSIGNMENT, Annual, Call, Note]","[Case Subject, NEW CASE, Case #, Jeffrey Epstein case, Case, Case tt, Jeffrey E. Epstein Case, https://na4.salesforce.com/5006000000VDvNf Case #, Case U, https:/ Case]","[assigned, kyc, assignment, note, case, notification, dbforcepb, customer, pursuant, confidential]"
4,3,"[alert, compliance, pcr, jeffrey, epstein, kyc, attached, jj, deutsche, aml]","[Jeffrey, Epstein, Alert, PCR, KYC, Deutsche, York, Wealth, Management, Company]","[Jeffrey Edward Epstein, Jeffrey Epstein, Jeffrey S. Epstein, Jeffrey S Epstein, Jeffrey E. Epstein, Jeffrey Epstein Ill, JJ JJ Litchford Associate Banker Deutsche Bank Trust Company Americas Deutsche Asset, PCR Alert, JJ Litchford Associate Banker Deutsche Bank Trust Company Americas Deutsche Asset, JJ JJ Litchford Associate Banker Deutsche Bark Trust Company Americas Deutsche Asset .5 Wealth Management]","[alerts, alert, kyc, epstein, approval, cleared, aml, pcr, confidential, compliance]"
5,4,"[llc, 00, td, grat, wanek, forsythe, trust, kati, wagner, shari]","[LLC, GRAT, TRUST, KATI, SOUTHERN, WANEK, SHARI, WAGNER, WANEK-FORSYTHE, COMPANY]","[| | | |, | | |, | | | | CRW, | | | | SSW, | | | | SHARI, | | | | PLAN D, | | | CODE, | 5,284,956.19 | | | CRW, | | | | NOEL VOLPE, | | | KATI]","[fed, deposit, funds, transactions, amount, report, equibase, balances, holdings, confidential]"
6,5,"[lexington, 4th, meeting, 10022, associates, trading, hbrk, kahn, 575, richard]","[Lexington, Avenue, Floor, Kahn, Associates, HBRK, York, Richard, trading, meeting]","[Richard Kahn HBRK Associates Inc., trading meeting, Lexington Avenue 4th Floor New York, Richard Kahn HBRK Associates Inc, Floor New York, Richard Kahn FIBRK Associates Inc., Richard Kahn, New York, tel fax cell Re, 575 Lexington Avenue 4th Floor New York NY]","[trading, meeting, hbrk, confirm, kahn, date, fibrk, received, tuesday, thanks]"
7,6,"[financial, southern, llc, edi, 127608, sgr, fw, reassign, nina, needed]","[Southern, Financial, LLC, REASSIGN, Nina, Tona, NEEDED, Zack, URGENT, EDI]","[Southern Financial LLC, SOUTHERN FINANCIAL LLC, *REASSIGN NEEDED***FW, REASSIGN NEEDED FW, Nina Tona, Nina Tona RE, LLC, Kumar Sambhav Kumar, REASSIGN NEEDED***FW, Nina Tona Fw]","[responded, urgent, reassign, confirm, fed, contact, southern, revised, securities, onb]"
8